# Speech - Sarawgi et al. 2020 on ADReSS
**Spec:** PLAN.md B1, FINDINGS 5, 7. **Runs on:** Colab + Google Drive (ADReSS must not go to Kaggle, FINDINGS 7). Not run yet.

Put the repo's `common/` and `speech/` folders in `MyDrive/Alzheimers/` and the ADReSS **train** folder in `MyDrive/Alzheimers/ADReSS-IS2020-train/`. Set the runtime to a GPU. Do NOT use Colab's Gemini / AI assistant with the data loaded; keep the files on your own Drive only.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opensmile scikit-learn
import sys, numpy as np, pandas as pd, torch, pathlib
REPO = '/content/drive/MyDrive/Alzheimers'     # folder containing common/ and speech/
DATA = REPO + '/ADReSS-IS2020-train/ADReSS-IS2020-data/train'   # EDIT if your folder differs
sys.path.insert(0, REPO)
from speech.features import get_pause_features, get_intervention_features
from speech.models import MODELS
from speech.cv import cross_validate
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

## 1. Shape check (run first)

In [ ]:
for name, shape in [('disfluency',(4,11)),('acoustic',(4,21)),('interventions',(4,32,3))]:
    m = MODELS[name](); m.train()
    out = m(torch.randn(*shape)); loss = out.sum() + m.penalty(); loss.backward()
    print(name, tuple(out.shape), 'ok')

> **SEND BACK TO CLAUDE:**
> the three lines printed (`disfluency ... ok`, etc.). If it errors: the last 10-15 lines of the error. STOP there.

## 2. Labels and features (108 subjects)

In [ ]:
root = pathlib.Path(DATA); rows = []
for grp, label in [('cc', 0), ('cd', 1)]:
    for cha in sorted((root/'transcription'/grp).glob('*.cha')):
        rows.append(dict(id=cha.stem, label=label, cha=cha, wav=root/'Full_wave_enhanced_audio'/grp/f'{cha.stem}.wav'))
df = pd.DataFrame(rows); print(df.label.value_counts().to_dict(), 'n =', len(df))   # expect 54/54
disf = np.stack([get_pause_features(r.cha, r.wav) for r in df.itertuples()])
inter = np.stack([get_intervention_features(r.cha) for r in df.itertuples()])
print(disf.shape, inter.shape, 'any NaN/inf:', not np.isfinite(disf).all())

> **SEND BACK TO CLAUDE:**
> the one printed line with the label counts and n (expect 54/54, n = 108) and the shape line.

## 3. ComParE (6,373 functionals per file)
[OURS] `opensmile-python` `ComParE_2016` stands in for the official `SMILExtract -C IS13_ComParE.conf`
(same 6,373 feature count; exact equivalence not verified). Scaler + PCA(21) are fitted per fold inside `cross_validate`.

In [ ]:
import opensmile
smile = opensmile.Smile(feature_set=opensmile.FeatureSet.ComParE_2016, feature_level=opensmile.FeatureLevel.Functionals)
acoustic_raw = np.stack([smile.process_file(str(w)).values[0] for w in df.wav]); print(acoustic_raw.shape)  # (108, 6373)

> **SEND BACK TO CLAUDE:**
> the printed shape, expect (108, 6373). Slow cell - a few minutes is normal.

## 4. 5-fold CV (batch 8, Adam lr 0.01; inner-split checkpointing [OURS])

In [ ]:
feats = dict(disfluency=disf, acoustic_raw=acoustic_raw, interventions=inter)
oof, fold_of = cross_validate(feats, df.label.values, seed=0, device=DEVICE)

> **SEND BACK TO CLAUDE:**
> nothing unless it errors (then the last 10-15 lines). It runs for a while.

## 5. Metrics — subject-level bootstrap, 95% CI, n and protocol

In [ ]:
from common.metrics import balanced_acc_ci, f1_ci, auc_ci
y = df.label.values; g = df.id.values; proto = '5-fold CV, n=108, stratified, seed 0'
for k, v in oof.items():
    pred = (v >= 0.5).astype(int)
    print(k); print('  ', balanced_acc_ci(y, pred, proto, g)); print('  ', f1_ci(y, pred, proto, g))
    if k != 'ensemble_hard_vote': print('  ', auc_ci(y, v, proto, g))
pd.DataFrame(dict(id=df.id, label=y, fold=fold_of, **oof)).to_csv(REPO + '/speech_oof.csv', index=False)

> **SEND BACK TO CLAUDE:**
> EVERYTHING this cell prints (balanced accuracy / F1 / AUC lines for each network and the ensemble).